# Claude Inference

Clean notebook for Claude-based ratings or critiques runs using direct one-by-one inference.

## Imports And Notebook Root

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd

ROOT_CODE_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "utils").exists())
NOTEBOOK_HOME = ROOT_CODE_DIR / "experiment_notebooks" / "Claude_Pipeline"
os.chdir(NOTEBOOK_HOME)

if str(ROOT_CODE_DIR) not in sys.path:
    sys.path.append(str(ROOT_CODE_DIR))

from utils.llm_evaluation_utils import *
from utils.prompt_builder import build_rating_prompt, build_critique_prompt
from utils.models_setup import setup_anthropic, query_claude_model
from utils.data_setup import get_dataset_file_path, prepare_project_data

ROOT_CODE_DIR, NOTEBOOK_HOME


## Config And Constants

In [ ]:
TASK_SUBSET = "1000_tasks"      # "all_tasks" | "1000_tasks" | "50_tasks"
PROMPTING_TYPE = "zero_shot"   # "zero_shot" | "few_shot"
STAGE = "ratings"              # "ratings" | "critiques"
NUM_TRIALS = 1
MAX_TOKENS = 30000
TEMPERATURE = 1
MODEL_VERSION = "claude-sonnet-4-20250514"

TASK_SUBSET_MAP = {
    "all_tasks": "all",
    "1000_tasks": "1000",
    "50_tasks": "50",
}
DATA_TASK_SUBSET = TASK_SUBSET_MAP[TASK_SUBSET]

if STAGE == "ratings":
    SYSTEM_MESSAGE = RATING_SYSTEM_MESSAGE
    build_prompt_fn = build_rating_prompt
    update_results_df_fn = update_rating_in_df
elif STAGE == "critiques":
    SYSTEM_MESSAGE = CRITIQUES_SYSTEM_MESSAGE
    build_prompt_fn = build_critique_prompt
    update_results_df_fn = update_critiques_in_df
else:
    raise ValueError(f"Invalid stage: {STAGE}")

client, cfg = setup_anthropic(
    model=MODEL_VERSION,
    system_message=SYSTEM_MESSAGE,
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS,
)

cfg


## Data Loading

In [ ]:
uicrit_file, base64_screens_file, few_shot_samples_file = get_dataset_file_path(STAGE)

uicrit_df = pd.read_parquet(uicrit_file)
base64_screens_df = pd.read_parquet(base64_screens_file)

if few_shot_samples_file:
    few_shot_samples_df = pd.read_parquet(few_shot_samples_file)
else:
    few_shot_samples_df = None

print("UICrit rows:", len(uicrit_df))
print("Screens rows:", len(base64_screens_df))


## Load Responses DataFrame

In [ ]:
responses_df, results_paths = prepare_project_data(
    model_short=cfg["model_short"],
    num_trials=NUM_TRIALS,
    task_subset=DATA_TASK_SUBSET,
    shots=PROMPTING_TYPE,
    stage=STAGE,
)

text_responses_jsonl_file = results_paths["results_jsonl"]
model_results_file = results_paths["results_parquet"]

print("Responses shape:", responses_df.shape)
responses_df.head(2)


## Optional Test Sample

In [ ]:
# Uncomment to run on a tiny subset first.
# responses_df = responses_df.head(5).copy()
# responses_df.head()


## Run Claude Inference

In [ ]:
query_args = dict(
    client=client,
    model_version=cfg["model_version"],
    max_tokens=MAX_TOKENS,
    temperature=TEMPERATURE,
    system=SYSTEM_MESSAGE,
    image_format="jpeg",
)

run_llm_inference(
    responses_df=responses_df,
    base64_screens_df=base64_screens_df,
    few_shot_samples_df=few_shot_samples_df,
    evaluation_aspects=EVALUATION_MAIN_ASPECTS,
    build_prompt_fn=build_prompt_fn,
    query_fn=query_claude_model,
    query_args=query_args,
    save_jsonl_fn=save_response_text,
    update_results_df_fn=update_results_df_fn,
    output_jsonl=text_responses_jsonl_file,
    guidelines=GUIDELINES,
    prompting_type=PROMPTING_TYPE,
    requests_per_minute=cfg["rpm"],
    stage=STAGE,
)


## Explore Results

In [ ]:
responses_df.head()


In [ ]:
if STAGE == "critiques":
    columns_with_none = (responses_df.isna() | (responses_df == "")).sum()
else:
    columns_with_none = responses_df[EVALUATION_FIVE_ASPECTS].isna().sum()

columns_with_none


In [ ]:
if STAGE == "critiques":
    rows_with_none = responses_df[responses_df["critiques"].isna() | (responses_df["critiques"] == "")]
else:
    rows_with_none = responses_df[responses_df[EVALUATION_FIVE_ASPECTS].isna().any(axis=1)]

rows_with_none.head()


In [ ]:
if STAGE == "critiques":
    missing = (responses_df["critiques"].isna() | (responses_df["critiques"] == "")).sum()
else:
    missing = responses_df[EVALUATION_FIVE_ASPECTS].isna().any(axis=1).sum()

print("rows:", len(responses_df))
print("unique screen_task_id:", responses_df["screen_task_id"].nunique())
print("missing outputs:", missing)


## Save Results

In [ ]:
responses_df.to_parquet(model_results_file, index=False)
print("Saved:", model_results_file)
